# 08 - Top-3 Training, Ensembles, Uncertainty

## Goal
- train 3 best hyperparameter sets from Optuna,
- for each set train 5 fold-models (TimeSeriesSplit),
- run test predictions using 5 models per set,
- build ensemble predictions and uncertainty estimates,
- save metrics and prediction artifacts.


In [1]:
import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import entropy
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter


In [2]:
try:
    from stylin import InfoDisplayStyler
    styler = InfoDisplayStyler()
except Exception:
    class _FallbackStyler:
        def style_me(self, obj, title=None):
            if title:
                print(f"\n=== {title} ===")
            display(obj)

        def show_line(self, *args, sep=' ', title=None):
            text = sep.join(str(a) for a in args).strip()
            if title:
                print(f"{title}: {text}")
            else:
                print(text)

        def show_meta(self, data):
            print('shape:', getattr(data, 'shape', None))
            display(data.head(3) if hasattr(data, 'head') else data)

    styler = _FallbackStyler()


In [3]:
DATA_PATH = '../../data/raw/ethusdt_1h.csv'
CLEAN_PATH = '../../data/clean/'
LABELED_PATH = '../../data/labeled/'
PROCESSED_PATH = '../../data/processed/'
REPORTS_PATH = '../../reports/'
CHECKPOINTS_PATH = '../../checkpoints/'
LOGS_PATH = '../../logs/'

TRAIN_SELECTED_FILE = PROCESSED_PATH + 'train_selected.csv'
VAL_SELECTED_FILE = PROCESSED_PATH + 'val_selected.csv'
TEST_SELECTED_FILE = PROCESSED_PATH + 'test_selected.csv'
SELECTED_FEATURES_FILE = PROCESSED_PATH + 'selected_feature_columns.csv'
TOP3_FILE = REPORTS_PATH + 'optuna_top3.csv'

assert os.path.exists(TRAIN_SELECTED_FILE), 'Missing train_selected.csv. Run notebook 04 first.'
assert os.path.exists(VAL_SELECTED_FILE), 'Missing val_selected.csv. Run notebook 04 first.'
assert os.path.exists(TEST_SELECTED_FILE), 'Missing test_selected.csv. Run notebook 04 first.'
assert os.path.exists(SELECTED_FEATURES_FILE), 'Missing selected_feature_columns.csv. Run notebook 04 first.'
assert os.path.exists(TOP3_FILE), 'Missing optuna_top3.csv. Run notebook 07 first.'

os.makedirs(REPORTS_PATH, exist_ok=True)
os.makedirs(REPORTS_PATH + 'predictions/', exist_ok=True)
os.makedirs(CHECKPOINTS_PATH + 'top3_cv/', exist_ok=True)
os.makedirs(LOGS_PATH + 'tensorboard/top3_cv/', exist_ok=True)

train_df = pd.read_csv(TRAIN_SELECTED_FILE)
val_df = pd.read_csv(VAL_SELECTED_FILE)
test_df = pd.read_csv(TEST_SELECTED_FILE)
selected_features = pd.read_csv(SELECTED_FEATURES_FILE)['feature_column'].tolist()
top3_df = pd.read_csv(TOP3_FILE)

for df in [train_df, val_df, test_df]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')

test_df = test_df.sort_values('timestamp').reset_index(drop=True)
train_val_df = pd.concat([train_df, val_df], axis=0).sort_values('timestamp').reset_index(drop=True)

setup_report = pd.DataFrame({
    'dataset': ['train_val', 'test', 'top3_trials'],
    'rows': [len(train_val_df), len(test_df), len(top3_df)],
    'num_features': [len(selected_features), len(selected_features), np.nan],
})
styler.style_me(setup_report, title='Notebook 08 input setup')
styler.style_me(top3_df[['rank', 'trial_number', 'value']], title='Top-3 trials')


,dataset,rows,num_features
0,train_val,39263,38.000000
1,test,6939,38.000000
2,top3_trials,3,nan


,rank,trial_number,value
0,1,21,0.396377
1,2,15,0.394122
2,3,30,0.394113


## 1. Utilities: model, training, inference


In [4]:
class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def build_activation(name: str) -> nn.Module:
    name = str(name).lower()
    if name == 'relu':
        return nn.ReLU()
    if name == 'gelu':
        return nn.GELU()
    if name == 'leaky_relu':
        return nn.LeakyReLU(negative_slope=0.1)
    raise ValueError(f'Unsupported activation: {name}')


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], num_classes: int, activation: str, dropout: float, use_batchnorm: bool):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(build_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def get_scaler(scaler_type: str):
    scaler_type = str(scaler_type).lower()
    if scaler_type == 'standard':
        return StandardScaler()
    if scaler_type == 'robust':
        return RobustScaler()
    raise ValueError(f'Unsupported scaler_type: {scaler_type}')


def build_optimizer(name: str, model: nn.Module, lr: float, weight_decay: float):
    name = str(name).lower()
    if name == 'adam':
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == 'adamw':
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    raise ValueError(f'Unsupported optimizer: {name}')


def compute_class_weights(y: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(y, minlength=num_classes)
    weights = len(y) / (num_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)


def train_one_epoch(model, dataloader, optimizer, criterion, device, clip_grad_norm=None):
    model.train()
    running_loss = 0.0
    n_batches = 0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()

        if clip_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        optimizer.step()
        running_loss += float(loss.item())
        n_batches += 1
    return running_loss / max(1, n_batches)


@torch.no_grad()
def evaluate_loader(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    n_batches = 0
    all_preds = []
    all_targets = []
    all_probs = []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

        running_loss += float(loss.item())
        n_batches += 1

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs)

    metrics = {
        'loss': running_loss / max(1, n_batches),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
    }
    return metrics, y_prob, y_pred, y_true


def parse_top3_row(row: pd.Series) -> dict:
    n_layers = int(row['n_layers'])
    hidden_dims = []
    for i in range(1, n_layers + 1):
        key = f'hidden_dim_{i}'
        if key not in row.index or pd.isna(row[key]):
            raise ValueError(f'Missing {key} for n_layers={n_layers}')
        hidden_dims.append(int(row[key]))

    return {
        'n_layers': n_layers,
        'hidden_dims': hidden_dims,
        'dropout': float(row['dropout']),
        'activation': str(row['activation']),
        'use_batchnorm': bool(row['use_batchnorm']),
        'optimizer': str(row['optimizer']),
        'lr': float(row['lr']),
        'weight_decay': float(row['weight_decay']),
        'batch_size': int(row['batch_size']),
        'scaler_type': str(row['scaler_type']),
        'clip_grad_norm': float(row['clip_grad_norm']),
    }


## 2. Top-3 x 5-fold training and test prediction


In [5]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_SPLITS = 5
EPOCHS = 30
PATIENCE = 8

X_tv = train_val_df[selected_features].to_numpy(dtype=np.float32)
y_tv = train_val_df['target'].to_numpy(dtype=np.int64)
X_test = test_df[selected_features].to_numpy(dtype=np.float32)
y_test = test_df['target'].to_numpy(dtype=np.int64)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')

all_ensemble_rows = []
metrics_rows = []

for _, top_row in top3_df.sort_values('rank').iterrows():
    rank = int(top_row['rank'])
    trial_number = int(top_row['trial_number'])
    params = parse_top3_row(top_row)

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_probabilities = []
    fold_metric_rows = []

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_tv), start=1):
        X_train_fold, y_train_fold = X_tv[train_idx], y_tv[train_idx]
        X_val_fold, y_val_fold = X_tv[val_idx], y_tv[val_idx]

        scaler = get_scaler(params['scaler_type'])
        X_train_scaled = scaler.fit_transform(X_train_fold).astype(np.float32)
        X_val_scaled = scaler.transform(X_val_fold).astype(np.float32)
        X_test_scaled = scaler.transform(X_test).astype(np.float32)

        train_loader = DataLoader(
            TabularDataset(X_train_scaled, y_train_fold),
            batch_size=params['batch_size'],
            shuffle=True,
            drop_last=True,
            num_workers=0,
        )
        val_loader = DataLoader(
            TabularDataset(X_val_scaled, y_val_fold),
            batch_size=params['batch_size'],
            shuffle=False,
            num_workers=0,
        )
        test_loader = DataLoader(
            TabularDataset(X_test_scaled, y_test),
            batch_size=params['batch_size'],
            shuffle=False,
            num_workers=0,
        )

        model = MLPClassifier(
            input_dim=X_train_scaled.shape[1],
            hidden_dims=params['hidden_dims'],
            num_classes=len(np.unique(y_tv)),
            activation=params['activation'],
            dropout=params['dropout'],
            use_batchnorm=params['use_batchnorm'],
        ).to(DEVICE)

        class_weights = compute_class_weights(y_train_fold, num_classes=len(np.unique(y_tv))).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = build_optimizer(params['optimizer'], model, params['lr'], params['weight_decay'])

        best_f1 = -np.inf
        bad_epochs = 0

        tb_dir = LOGS_PATH + f'tensorboard/top3_cv/run_{run_id}_rank{rank}_fold{fold_idx}'
        writer = SummaryWriter(log_dir=tb_dir)

        ckpt_path = CHECKPOINTS_PATH + f'top3_cv/run_{run_id}_rank{rank}_fold{fold_idx}.pt'

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, params['clip_grad_norm'])
            val_metrics, _, _, _ = evaluate_loader(model, val_loader, criterion, DEVICE)

            writer.add_scalar('loss/train', train_loss, epoch)
            writer.add_scalar('loss/val', val_metrics['loss'], epoch)
            writer.add_scalar('metrics/f1_macro', val_metrics['f1_macro'], epoch)

            if val_metrics['f1_macro'] > best_f1:
                best_f1 = val_metrics['f1_macro']
                bad_epochs = 0
                torch.save(
                    {
                        'model_state_dict': model.state_dict(),
                        'params': params,
                        'rank': rank,
                        'trial_number': trial_number,
                        'fold': fold_idx,
                        'best_val_f1': best_f1,
                    },
                    ckpt_path,
                )
            else:
                bad_epochs += 1
                if bad_epochs >= PATIENCE:
                    break

        writer.close()

        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])

        val_metrics, _, _, _ = evaluate_loader(model, val_loader, criterion, DEVICE)
        test_metrics, test_probs, test_preds, test_true = evaluate_loader(model, test_loader, criterion, DEVICE)

        fold_probabilities.append(test_probs)

        fold_metric_rows.append({
            'rank': rank,
            'trial_number': trial_number,
            'fold': fold_idx,
            'best_val_f1': float(ckpt['best_val_f1']),
            'test_f1_macro': float(test_metrics['f1_macro']),
            'test_balanced_accuracy': float(test_metrics['balanced_accuracy']),
            'test_accuracy': float(test_metrics['accuracy']),
        })

        model_pred_df = pd.DataFrame({
            'timestamp': test_df['timestamp'].astype(str),
            'y_true': test_true,
            'y_pred': test_preds,
            'prob_sell': test_probs[:, 0],
            'prob_hold': test_probs[:, 1],
            'prob_buy': test_probs[:, 2],
        })
        model_pred_path = REPORTS_PATH + f'predictions/run_{run_id}_rank{rank}_fold{fold_idx}_test_predictions.csv'
        model_pred_df.to_csv(model_pred_path, index=False)

    stacked = np.stack(fold_probabilities, axis=0)  # [5, n_test, n_classes]
    ensemble_probs = stacked.mean(axis=0)
    ensemble_preds = np.argmax(ensemble_probs, axis=1)

    ensemble_metrics = {
        'rank': rank,
        'trial_number': trial_number,
        'cv_score_from_optuna': float(top_row['value']),
        'test_f1_macro': float(f1_score(y_test, ensemble_preds, average='macro', zero_division=0)),
        'test_balanced_accuracy': float(balanced_accuracy_score(y_test, ensemble_preds)),
        'test_accuracy': float(accuracy_score(y_test, ensemble_preds)),
        'test_precision_macro': float(precision_score(y_test, ensemble_preds, average='macro', zero_division=0)),
        'test_recall_macro': float(recall_score(y_test, ensemble_preds, average='macro', zero_division=0)),
    }
    metrics_rows.append(ensemble_metrics)

    # Uncertainty
    prob_std = stacked.std(axis=0)
    mean_prob_std = prob_std.mean(axis=1)
    ent = entropy(ensemble_probs.T)
    sorted_probs = np.sort(ensemble_probs, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    ensemble_df = pd.DataFrame({
        'timestamp': test_df['timestamp'].astype(str),
        'y_true': y_test,
        'y_pred': ensemble_preds,
        'prob_sell': ensemble_probs[:, 0],
        'prob_hold': ensemble_probs[:, 1],
        'prob_buy': ensemble_probs[:, 2],
        'uncertainty_entropy': ent,
        'uncertainty_margin': margin,
        'uncertainty_mean_prob_std': mean_prob_std,
    })

    ensemble_path = REPORTS_PATH + f'predictions/run_{run_id}_rank{rank}_ensemble_test_predictions.csv'
    ensemble_df.to_csv(ensemble_path, index=False)

    all_ensemble_rows.extend(fold_metric_rows)


## 3. Save final reports


In [6]:
metrics_df = pd.DataFrame(metrics_rows).sort_values('rank')
fold_metrics_df = pd.DataFrame(all_ensemble_rows).sort_values(['rank', 'fold'])

metrics_path = REPORTS_PATH + f'top3_ensemble_metrics_{run_id}.csv'
fold_metrics_path = REPORTS_PATH + f'top3_fold_metrics_{run_id}.csv'
summary_path = REPORTS_PATH + f'top3_ensemble_summary_{run_id}.json'
latest_metrics_alias = REPORTS_PATH + 'top3_ensemble_metrics.csv'
latest_summary_alias = REPORTS_PATH + 'top3_ensemble_summary.json'

metrics_df.to_csv(metrics_path, index=False)
metrics_df.to_csv(latest_metrics_alias, index=False)
fold_metrics_df.to_csv(fold_metrics_path, index=False)

summary = {
    'run_id': run_id,
    'n_sets': int(len(metrics_df)),
    'n_folds_per_set': int(N_SPLITS),
    'metrics_file': metrics_path,
    'fold_metrics_file': fold_metrics_path,
    'best_set_by_test_f1': metrics_df.sort_values('test_f1_macro', ascending=False).iloc[0].to_dict() if len(metrics_df) else None,
}
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with open(latest_summary_alias, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

styler.style_me(metrics_df, title='Top-3 ensemble metrics on test')
styler.style_me(fold_metrics_df, title='Fold-level metrics (5 models per set)')
styler.show_line(metrics_path, title='Saved metrics file')
styler.show_line(summary_path, title='Saved summary file')


,rank,trial_number,cv_score_from_optuna,test_f1_macro,test_balanced_accuracy,test_accuracy,test_precision_macro,test_recall_macro
0,1,21,0.396377,0.364144,0.378471,0.366912,0.410450,0.378471
1,2,15,0.394122,0.378635,0.392899,0.383196,0.418077,0.392899
2,3,30,0.394113,0.355021,0.374701,0.360715,0.390063,0.374701


,rank,trial_number,fold,best_val_f1,test_f1_macro,test_balanced_accuracy,test_accuracy
0,1,21,1,0.369419,0.348190,0.376918,0.378008
1,1,21,2,0.385455,0.352506,0.367522,0.359706
2,1,21,3,0.404021,0.339037,0.367827,0.343854
3,1,21,4,0.418753,0.348183,0.380885,0.349330
4,1,21,5,0.391174,0.385986,0.398269,0.391699
5,2,15,1,0.365138,0.377571,0.381640,0.392276
6,2,15,2,0.399695,0.384895,0.392027,0.408704
7,2,15,3,0.410573,0.344815,0.373552,0.346015
8,2,15,4,0.419464,0.332638,0.374160,0.339963
9,2,15,5,0.371725,0.352812,0.386783,0.356680


## 4. Quick uncertainty summary


In [7]:
summary_rows = []
for rank in sorted(metrics_df['rank'].unique()):
    p = REPORTS_PATH + f'predictions/run_{run_id}_rank{rank}_ensemble_test_predictions.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        summary_rows.append({
            'rank': rank,
            'mean_entropy': float(df['uncertainty_entropy'].mean()),
            'mean_margin': float(df['uncertainty_margin'].mean()),
            'mean_prob_std': float(df['uncertainty_mean_prob_std'].mean()),
        })

uncertainty_summary_df = pd.DataFrame(summary_rows).sort_values('rank')
styler.style_me(uncertainty_summary_df, title='Uncertainty summary per top-set')


,rank,mean_entropy,mean_margin,mean_prob_std
0,1,1.069630,0.085410,0.055212
1,2,1.067161,0.077924,0.044761
2,3,1.055580,0.133790,0.123206


## 5. Final conclusions
- Dla kazdego z 3 najlepszych zestawow hiperparametrow wytrenowano 5 modeli foldowych i wykonano ensemble na zbiorze testowym.
- Najlepszy ensemble na te?cie to **rank 2 / trial 15**:
  - `test_f1_macro = 0.3786`
  - `test_balanced_accuracy = 0.3929`
  - `test_accuracy = 0.3832`
- Pozostale zestawy:
  - rank 1 / trial 21: `f1_macro = 0.3641`
  - rank 3 / trial 30: `f1_macro = 0.3550`
- Analiza niepewnosci pokazuje, ze rank 3 ma najwyzszy sredni `margin` (~0.1338), ale mimo to osiaga najslabsze metryki klasyfikacyjne.
- Wszystkie artefakty do testow statystycznych zostaly zapisane (`top3_ensemble_metrics.csv`, `top3_ensemble_summary.json`, pliki predykcji per rank), wiec notebook 09 moze porownac ensemble'e statystycznie.
